## Гружу датасет в s3

In [2]:
import boto3
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

In [3]:
s3 = boto3.client(
    service_name="s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id="YCAJEK6EmbclThg_wry3OwCsF",
    aws_secret_access_key="YCORGYliOZhkb3VJRLOdQDxBuFNV_46YsjZZJ3XX",
)

In [4]:
i = 0
for key in s3.list_objects(Bucket='sneakers-hse-images-test')['Contents']:
    print(key['Key'])
    i += 1
    if i == 5:
        break

real-labeled-sneakers-dataset/.DS_Store
real-labeled-sneakers-dataset/Kari Кеды/.DS_Store
real-labeled-sneakers-dataset/Kari Кеды/0.jpeg
real-labeled-sneakers-dataset/Kari Кеды/1.jpeg
real-labeled-sneakers-dataset/Kari Кеды/10.jpeg


In [5]:
def upload_folder_to_s3(
    local_folder: str,
    bucket_name: str,
    s3_prefix: str = ""
):
    """
    Загружает папку с сохранением вложенной структуры
    """
    for root, _, files in os.walk(local_folder):
        for file in files:
            local_path = os.path.join(root, file)

            relative_path = os.path.relpath(local_path, local_folder)
            s3_key = os.path.join(s3_prefix, relative_path).replace("\\", "/")

            s3.upload_file(local_path, bucket_name, s3_key)
            print(f"Uploaded: {s3_key}")

In [6]:
def _upload_one(local_path, bucket_name, s3_key):
    s3.upload_file(local_path, bucket_name, s3_key)
    return s3_key


def upload_folder_to_s3_parallel(
    local_folder: str,
    bucket_name: str,
    s3_prefix: str = "",
    max_workers: int = 16
):
    """
    Параллельная загрузка папки в S3
    """
    tasks = []

    for root, _, files in os.walk(local_folder):
        for file in files:
            local_path = os.path.join(root, file)

            relative_path = os.path.relpath(local_path, local_folder)
            s3_key = os.path.join(s3_prefix, relative_path).replace("\\", "/")

            tasks.append((local_path, s3_key))

    print(f"Total files: {len(tasks)}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(_upload_one, local_path, bucket_name, s3_key)
            for local_path, s3_key in tasks
        ]

        for future in as_completed(futures):
            try:
                s3_key = future.result()
                print(f"Uploaded: {s3_key}")
            except Exception as e:
                print(f"Error: {e}")

In [7]:
upload_folder_to_s3_parallel(
    local_folder="../../data/03_yolo_preprocessed",
    bucket_name="sneakers-hse-images-test",
    s3_prefix="yolo_preprocessed",
    max_workers=144
)

Total files: 11268
Uploaded: yolo_preprocessed/.gitkeep
Uploaded: yolo_preprocessed/test_images.csv
Uploaded: yolo_preprocessed/search-dataset-images/Vans Кеды Upland/clothing_0_264.jpeg
Uploaded: yolo_preprocessed/search-dataset-images/Vans Кеды Upland/orig_72.jpeg
Uploaded: yolo_preprocessed/search-dataset-images/Vans Кеды Upland/clothing_0_57.jpeg
Uploaded: yolo_preprocessed/search-dataset-images/Vans Кеды Upland/clothing_0_116.jpeg
Uploaded: yolo_preprocessed/search-dataset-images/Vans Кеды Upland/orig_139.jpeg
Uploaded: yolo_preprocessed/search-dataset-images/Vans Кеды Upland/clothing_0_37.jpeg
Uploaded: yolo_preprocessed/search-dataset-images/Vans Кеды Upland/clothing_0_0.jpeg
Uploaded: yolo_preprocessed/search-dataset-images/Vans Кеды Upland/clothing_0_194.jpeg
Uploaded: yolo_preprocessed/search-dataset-images/Vans Кеды Upland/orig_115.jpeg
Uploaded: yolo_preprocessed/search-dataset-images/Vans Кеды Upland/orig_267.jpeg
Uploaded: yolo_preprocessed/search-dataset-images/Vans Кеды